# Customer, Product and Profitability Performance Analysis
## Data Cleaning & Financial Validation 

### Objective
- This notebook performs initial data inspection, data-quality validation, financial consistency checks, anomly detection and preparation of the cleab analytical dataset.

### Business Context
The analysis aims to understand customer value, product profitability, regional performance and discount-driven margin erosion within supply chain operations.

In [1]:
# Import libraries

import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
# Load the raw dataset

DATA_PATH = "../data/raw/APL_Logistics.csv"

df = pd.read_csv(
    DATA_PATH,
    encoding="latin1"
)

print(f"Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully.
Rows: 180,519
Columns: 40


In [3]:
# Inspect the structure 

df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Fname,Customer Id,Customer Lname,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,Product Price,Shipping Mode
0,DEBIT,6,4,159.69,472.45,Late delivery,1,9,Cardio Equipment,Brownsville,EE. UU.,Richard,1,Hernandez,Consumer,TX,6303 Heather Plaza,"78,521.00",3,Footwear,25.95,-97.51,Pacific Asia,Mumbai,India,1,27.50,0.06,99.99,0.34,5,499.95,472.45,159.69,South Asia,Maharashtra,COMPLETE,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class
1,DEBIT,4,4,48.71,167.96,Shipping on time,0,29,Shop By Sport,Littleton,EE. UU.,Mary,2,Barrett,Consumer,CO,9526 Noble Embers Ridge,"80,126.00",5,Golf,38.38,-104.73,LATAM,San Pedro Sula,Honduras,2,31.99,0.16,39.99,0.29,5,199.95,167.96,48.71,Central America,Cortés,ON_HOLD,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class
2,DEBIT,4,4,87.36,181.99,Shipping on time,0,48,Water Sports,Littleton,EE. UU.,Mary,2,Barrett,Consumer,CO,9526 Noble Embers Ridge,"80,126.00",7,Fan Shop,38.38,-104.73,LATAM,San Pedro Sula,Honduras,2,18.00,0.09,199.99,0.48,1,199.99,181.99,87.36,Central America,Cortés,ON_HOLD,Pelican Sunstream 100 Kayak,199.99,Standard Class
3,DEBIT,6,4,-41.89,175.99,Late delivery,1,48,Water Sports,Littleton,EE. UU.,Mary,2,Barrett,Consumer,CO,9526 Noble Embers Ridge,"80,126.00",7,Fan Shop,38.38,-104.73,USCA,New York City,Estados Unidos,2,24.00,0.12,199.99,-0.24,1,199.99,175.99,-41.89,East of USA,Nueva York,COMPLETE,Pelican Sunstream 100 Kayak,199.99,Standard Class
4,DEBIT,6,4,10.00,40.00,Late delivery,1,24,Women's Apparel,Littleton,EE. UU.,Mary,2,Barrett,Consumer,CO,9526 Noble Embers Ridge,"80,126.00",5,Golf,38.38,-104.73,USCA,New York City,Estados Unidos,2,10.00,0.20,50.00,0.25,1,50.00,40.00,10.00,East of USA,Nueva York,COMPLETE,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class


In [4]:
df.tail()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Fname,Customer Id,Customer Lname,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,Product Price,Shipping Mode
180514,TRANSFER,6,4,-107.96,161.87,Late delivery,1,76,Women's Clothing,Caguas,Puerto Rico,Chava,20753,Velez,Consumer,PR,9861 Grand Elk Trace,725.00,4,Apparel,18.27,-66.04,Pacific Asia,Bekasi,Indonesia,20753,53.96,0.25,215.82,-0.67,1,215.82,161.87,-107.96,Southeast Asia,Java Occidental,PROCESSING,Summer dresses,215.82,Standard Class
180515,DEBIT,2,4,-126.56,172.66,Advance shipping,0,76,Women's Clothing,Plainfield,EE. UU.,Stacey,20754,Weber,Consumer,IL,5919 Hidden Anchor Port,"60,544.00",4,Apparel,41.61,-88.20,Pacific Asia,Bekasi,Indonesia,20754,43.16,0.20,215.82,-0.73,1,215.82,172.66,-126.56,Southeast Asia,Java Occidental,COMPLETE,Summer dresses,215.82,Standard Class
180516,DEBIT,3,4,91.25,314.64,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,Cally,20755,Holloway,Consumer,PR,5365 Noble Nectar Island,725.00,2,Fitness,18.25,-66.04,Pacific Asia,Bekasi,Indonesia,20755,13.11,0.04,327.75,0.29,1,327.75,314.64,91.25,Southeast Asia,Java Occidental,COMPLETE,Smart watch,327.75,Standard Class
180517,PAYMENT,4,4,1.53,10.91,Shipping on time,0,74,Toys,Berwyn,EE. UU.,Cherokee,20756,Callahan,Corporate,IL,3312 Harvest Horse Chase,"60,402.00",7,Fan Shop,41.83,-87.80,Pacific Asia,Bekasi,Indonesia,20756,0.63,0.06,11.54,0.14,1,11.54,10.91,1.53,Southeast Asia,Java Occidental,PENDING_PAYMENT,Toys,11.54,Standard Class
180518,CASH,1,0,9.79,34.98,Late delivery,1,75,Video Games,Miami,EE. UU.,Katelyn,20757,Gonzales,Corporate,FL,9299 Quiet Pioneer Forest,"33,126.00",9,Discs Shop,25.86,-80.20,Pacific Asia,Gold Coast,Australia,20757,4.77,0.12,39.75,0.28,1,39.75,34.98,9.79,Oceania,Queensland,CLOSED,Fighting video games,39.75,Same Day


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 40 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Fname                 180519 non-null  str    
 12  Customer Id                    180519 non

In [6]:
df.shape

(180519, 40)

In [7]:
# Column Inspection

columns_df = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isna().sum().values,
    "Missing %": (df.isna().mean()*100).round(2).values,
    "Unique Values": df.nunique().values
})

columns_df

,Column,Data Type,Missing Values,Missing %,Unique Values
0,Type,str,0,0.00,4
1,Days for shipping (real),int64,0,0.00,7
2,Days for shipment (scheduled),int64,0,0.00,4
3,Benefit per order,float64,0,0.00,21998
4,Sales per customer,float64,0,0.00,2927
5,Delivery Status,str,0,0.00,4
6,Late_delivery_risk,int64,0,0.00,2
7,Category Id,int64,0,0.00,51
8,Category Name,str,0,0.00,50
9,Customer City,str,0,0.00,563


In [8]:
# Missing value aanlysis

missing = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

Customer Lname      8
Customer Zipcode    3
dtype: int64

In [9]:
# Duplicate Validation 

duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count:,}")

Duplicate rows: 0


In [10]:
# Check Numerical Variables

numeric_columns = df.select_dtypes(include=np.number).columns

df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
Days for shipping (real),"180,519.00",3.50,1.62,0.00,2.00,3.00,5.00,6.00
Days for shipment (scheduled),"180,519.00",2.93,1.37,0.00,2.00,4.00,4.00,4.00
Benefit per order,"180,519.00",21.97,104.43,"-4,274.98",7.00,31.52,64.80,911.80
Sales per customer,"180,519.00",183.11,120.04,7.49,104.38,163.99,247.40,"1,939.99"
Late_delivery_risk,"180,519.00",0.55,0.50,0.00,0.00,1.00,1.00,1.00
Category Id,"180,519.00",31.85,15.64,2.00,18.00,29.00,45.00,76.00
Customer Id,"180,519.00","6,691.38","4,162.92",1.00,"3,258.50","6,457.00","9,779.00","20,757.00"
Customer Zipcode,"180,516.00","35,921.13","37,542.46",603.00,725.00,"19,380.00","78,207.00","99,205.00"
Department Id,"180,519.00",5.44,1.63,2.00,4.00,5.00,7.00,12.00
Latitude,"180,519.00",29.72,9.81,-33.94,18.27,33.14,39.28,48.78


## Financial Validation

### Validation 1: Product Price Consistency

In [11]:
# Dataset contains both "Product Price" & "Order Item Product Price"
price_check = np.isclose(
    df["Product Price"],
    df["Order Item Product Price"],
    atol=0.01
)

print(f"Matching records: {price_check.sum():,}")
print(f"Non-matching records: {(~price_check).sum():,}")
print(f"Match rate: {price_check.mean()*100:2f}%")

Matching records: 180,519
Non-matching records: 0
Match rate: 100.000000%


### Validation 2: Sales Calculation

In [12]:
# Sales = Product Price x Order Item Quantity

calculated_sales = (
    df["Order Item Product Price"] *
    df["Order Item Quantity"]
)

sales_check = np.isclose(
    df["Sales"],
    calculated_sales,
    atol=0.01
)

print(f"Matching records: {sales_check.sum():,}")
print(f"Non-matching records: {(~sales_check).sum():,}")
print(f"Match rate: {sales_check.mean() * 100:.2f}%")

Matching records: 180,519
Non-matching records: 0
Match rate: 100.00%


### Validation 3: Discount Calculation

In [13]:
# Order Item Total = Sales - Order Item Discount

calculated_total = (
    df["Sales"] - 
    df["Order Item Discount"]
)

total_check = np.isclose(
    df["Order Item Total"],
    calculated_total,
    atol=0.01
)

print(f"Matching records: {total_check.sum():,}")
print(f"Non-matching records: {(~total_check).sum():,}")
print(f"Match rate: {total_check.mean() * 100:.2f}%")

Matching records: 180,519
Non-matching records: 0
Match rate: 100.00%


### Validation 4: Profit Consistency 

In [14]:
# Dataset contains "Benefit Per order" & "Order Profit Per Order"

profit_check = np.isclose(
    df["Benefit per order"],
    df["Order Profit Per Order"],
    atol = 0.01
)

print(f"Matching records: {profit_check.sum():,}")
print(f"Non-matching records: {(~profit_check).sum():,}")
print(f"Match rate: {profit_check.mean() * 100: .2f}%")

Matching records: 180,519
Non-matching records: 0
Match rate:  100.00%


### Validation 5: Profit Ratio

In [15]:
# Order Item Profit Ratio = Order Profit Per Order / Order Item Total

calculated_profit_ratio = (
    df["Order Profit Per Order"] /
    df["Order Item Total"]
)

ratio_difference = (
    df["Order Item Profit Ratio"] - 
    calculated_profit_ratio
).abs()

print("Maximum absolute difference:",
      ratio_difference.max())

print("Records with difference > 0.01:",
      (ratio_difference > 0.01).sum())

Maximum absolute difference: 0.005505293551491874
Records with difference > 0.01: 0


In [16]:
# Check zero and negative financial values

financial_columns = [
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Benefit per order",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Product Price",
    "Product Price",
    "Order Item Quantity"
]

financial_quality = pd.DataFrame({
    "Zero Values": (df[financial_columns] == 0).sum(),
    "Negative Values": (df[financial_columns] < 0).sum(),
    "Missing Values": df[financial_columns].isna().sum()
})

financial_quality

,Zero Values,Negative Values,Missing Values
Sales,0,0,0
Order Item Total,0,0,0
Order Profit Per Order,1177,33784,0
Benefit per order,1177,33784,0
Order Item Discount,10028,0,0
Order Item Discount Rate,10028,0,0
Order Item Product Price,0,0,0
Product Price,0,0,0
Order Item Quantity,0,0,0


### Sales and Profit Overview

In [17]:
total_sales = df["Sales"].sum()
total_profit = df["Order Profit Per Order"].sum()
total_discount = df["Order Item Discount"].sum()

profit_margin = total_profit / total_sales

print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Total Discount: ${total_discount:,.2f}")
print(f"Profit Margin: {profit_margin:.2%}")

Total Sales: $36,784,734.31
Total Profit: $3,966,902.97
Total Discount: $3,730,378.40
Profit Margin: 10.78%


## Step 1B: Anomaly Investigation & Cleaning Rules

The pupose of this section is to distinguish genuine business observations from data-quality errors. Negative profitability is retained because loss-making transactions are relevant to the profitability analysis. 

In [18]:
# Profit distribution

profit_summary = df["Order Profit Per Order"].describe()

profit_summary

count   180,519.00
mean         21.97
std         104.43
min      -4,274.98
25%           7.00
50%          31.52
75%          64.80
max         911.80
Name: Order Profit Per Order, dtype: float64

In [19]:
print("Minimum Profit:",
      df["Order Profit Per Order"].min())

print("Maximum Profit:",
      df["Order Profit Per Order"].max())

print("Median Profit:",
      df["Order Profit Per Order"].median())

Minimum Profit: -4274.98
Maximum Profit: 911.8
Median Profit: 31.52


In [20]:
# Sales distribution

sales_summary = df["Sales"].describe()

sales_summary

count   180,519.00
mean        203.77
std         132.27
min           9.99
25%         119.98
50%         199.92
75%         299.95
max       1,999.99
Name: Sales, dtype: float64

In [21]:
print("Minimum Sales:",
      df["Sales"].min())

print("Maximum Sales:",
      df["Sales"].max())

print("Median Sales:",
      df["Sales"].median())

Minimum Sales: 9.99
Maximum Sales: 1999.99
Median Sales: 199.92


In [22]:
# Discount distribution

discount_summary = df["Order Item Discount Rate"].describe()

discount_summary

count   180,519.00
mean          0.10
std           0.07
min           0.00
25%           0.04
50%           0.10
75%           0.16
max           0.25
Name: Order Item Discount Rate, dtype: float64

In [23]:
print("Minimum Discount Rate:",
      df["Order Item Discount Rate"].min())

print("Maximum Discount Rate:",
      df["Order Item Discount Rate"].max())

print("Median Discount Rate:",
      df["Order Item Discount Rate"].median())

Minimum Discount Rate: 0.0
Maximum Discount Rate: 0.25
Median Discount Rate: 0.1


In [24]:
# Check discount logic

invalid_discount = df[
    df["Order Item Discount"] > df["Sales"]
]

print("Records where discount exceeds sales:",
      len(invalid_discount))

Records where discount exceeds sales: 0


In [25]:
invalid_discount_rate = df[
    (df["Order Item Discount Rate"] < 0) |
    (df["Order Item Discount Rate"] > 1)
]

print("Invalid discount rates:",
      len(invalid_discount_rate))

Invalid discount rates: 0


In [26]:
# Check quantity

invalid_quantity = df[
    df["Order Item Quantity"] <=0
]

print("Invalid quantities:",
      len(invalid_quantity))

Invalid quantities: 0


In [27]:
#Check geogrphic coordinates

invalid_latitude = df[
    (df["Latitude"] < -90) |
    (df["Latitude"] > 90)
]

invalid_longitude = df[
    (df["Longitude"] < -180) |
    (df["Longitude"] > 180)
]

print("Invalid latitude:", len(invalid_latitude))
print("Invalid longitude:", len(invalid_longitude))

Invalid latitude: 0
Invalid longitude: 0


In [28]:
# Check categorical values

categorical_columns = [
    "Type",
    "Delivery Status",
    "Category Name",
    "Customer Segment",
    "Market",
    "Order Region",
    "Order Status",
    "Shipping Mode"
    ]

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- Type ---
Type
DEBIT       69295
TRANSFER    49883
PAYMENT     41725
CASH        19616
Name: count, dtype: int64

--- Delivery Status ---
Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

--- Category Name ---
Category Name
Cleats                  24551
Men's Footwear          22246
Women's Apparel         21035
Indoor/Outdoor Games    19298
Fishing                 17325
Water Sports            15540
Camping & Hiking        13729
Cardio Equipment        12487
Shop By Sport           10984
Electronics              3156
Accessories              1780
Golf Balls               1475
Girls' Apparel           1201
Golf Gloves              1070
Trade-In                  974
Video Games               838
Children's Clothing       652
Women's Clothing          650
Baseball & Softball       632
Hockey                    614
Cameras                   592
Toys                      529
Golf Shoes 

In [29]:
# Check whitespace in text fields.

text_columns = df.select_dtypes(include="object").columns

whitespace_issues = {}

for col in text_columns:
    mask = df[col].astype(str).str.strip() != df[col].astype(str)
    count = mask.sum()

    if count > 0:
        whitespace_issues[col] = count

whitespace_issues

/var/folders/qj/27vs02yj54qgcsfcgp1rpks40000gn/T/ipykernel_12704/1231207986.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include="object").columns


{'Category Name': np.int64(1475),
 'Customer Lname': np.int64(8),
 'Customer Street': np.int64(1829),
 'Department Name': np.int64(362),
 'Order Region': np.int64(17925),
 'Product Name': np.int64(1774)}

## Text Standardisation

Leading and trailing whitespace was identified in several categorical and
customer/product text fields. These spaces can create duplicate category
values during aggregation and therefore will be removed.

Missing values will not be converted into strings.

In [31]:
# Identify text columns
text_columns = df.select_dtypes(include=["object", "string"]).columns

# Remove leading and trailing whitespace while preserving missing values
for col in text_columns:
    df[col] = df[col].str.strip()

In [34]:
# Recheck the whitespace

whitespace_issues_after = {}

for col in text_columns:
    mask = df[col].notna() & (df[col] != df[col].str.strip())
    count = mask.sum()

    if count > 0:
        whitespace_issues_after[col] = count

whitespace_issues_after

{}

In [30]:
#Check duplicate customer IDs

customer_count = df["Customer Id"].nunique()
order_count = len(df)

print(f"Unique customers: {customer_count:,}")
print(f"Total transactions: {order_count:,}")

Unique customers: 20,652
Total transactions: 180,519


In [ ]:
# Check categorical values again

categorical_columns = [
    "Type",
    "Delivery Status",
    "Category Name",
    "Customer Segment",
    "Market",
    "Order Region",
    "Order Status",
    "Shipping Mode"
]

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(f"Unique values: {df[col].nunique()}")
    print(df[col].value_counts(dropna=False).head(20))


--- Type ---
Unique values: 4
Type
DEBIT       69295
TRANSFER    49883
PAYMENT     41725
CASH        19616
Name: count, dtype: int64

--- Delivery Status ---
Unique values: 4
Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

--- Category Name ---
Unique values: 50
Category Name
Cleats                  24551
Men's Footwear          22246
Women's Apparel         21035
Indoor/Outdoor Games    19298
Fishing                 17325
Water Sports            15540
Camping & Hiking        13729
Cardio Equipment        12487
Shop By Sport           10984
Electronics              3156
Accessories              1780
Golf Balls               1475
Girls' Apparel           1201
Golf Gloves              1070
Trade-In                  974
Video Games               838
Children's Clothing       652
Women's Clothing          650
Baseball & Softball       632
Hockey                    614
Name: count, dtype:

In [ ]:
# Check product/category consistency

product_category_check = (
    df.groupby("Product Name")["Category Name"]
      .nunique()
      .sort_values(ascending=False)
)

product_category_check.head(20)

Product Name
Adult dog supplies                               1
Porcelain crafts                                 1
Team Golf New England Patriots Putter Grip       1
TaylorMade Women's RBZ SL Rescue                 1
TaylorMade White Smoke IN-12 Putter              1
TaylorMade 2014 Purelite Stand Bag               1
TYR Boys' Team Digi Jammer                       1
Summer dresses                                   1
Stiga Master Series ST3100 Competition Indoor    1
Sports Books                                     1
Smart watch                                      1
SOLE E35 Elliptical                              1
SOLE E25 Elliptical                              1
Rock music                                       1
Polar Loop Activity Tracker                      1
Team Golf San Francisco Giants Putter Grip       1
Polar FT4 Heart Rate Monitor                     1
Perfect Fitness Perfect Rip Deck                 1
Pelican Sunstream 100 Kayak                      1
Pelican Maverick 1

In [38]:
inconsistent_products = product_category_check[
    product_category_check > 1
]

print(
    f"Products associated with multiple categories: "
    f"{len(inconsistent_products):,}"
)

Products associated with multiple categories: 0


In [ ]:
# Department consistency

product_department_check = (
    df.groupby("Product Name")["Department Name"]
      .nunique()
      .sort_values(ascending=False)
)

print(
    f"Products associated with multiple departments: "
    f"{(product_department_check > 1).sum():,}"
)

Products associated with multiple departments: 0


In [ ]:
# Customer consistency

customer_segment_check = (
    df.groupby("Customer Id")["Customer Segment"]
      .nunique()
)

print(
    f"Customers associated with multiple segments: "
    f"{(customer_segment_check > 1).sum():,}"
)

Customers associated with multiple segments: 0


In [41]:
customer_country_check = (
    df.groupby("Customer Id")["Customer Country"]
      .nunique()
)

print(
    f"Customers associated with multiple countries: "
    f"{(customer_country_check > 1).sum():,}"
)

Customers associated with multiple countries: 0


## Export Clean Dataset

The validated and standardised transaction-level dataset is exported for
use in subsequent exploratory analysis and the Streamlit application.

In [43]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

clean_path = processed_dir / "apl_logistics_clean.csv"

df.to_csv(clean_path, index=False)

print(f"Clean dataset saved to: {clean_path}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Clean dataset saved to: ../data/processed/apl_logistics_clean.csv
Rows: 180,519
Columns: 40


In [44]:
df_clean = pd.read_csv(
    clean_path,
    encoding="latin1"
)

print("Reloaded successfully.")
print(f"Rows: {df_clean.shape[0]:,}")
print(f"Columns: {df_clean.shape[1]}")

Reloaded successfully.
Rows: 180,519
Columns: 40


In [45]:
print("Duplicate rows:", df_clean.duplicated().sum())
print("Missing values:", df_clean.isna().sum().sum())

Duplicate rows: 0
Missing values: 11
